In [15]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from PIL import Image

In [16]:
CSV_PATH = "../processing/master_with_paths.csv"
DATASET_ROOT = "../data/render-lighting"
ALLOWED_LIGHT_FOLDERS = {"Tri Lighting"}
MATERIAL = "PlasticGlossy"
BATCH = "Batch 1 - Cycles AGX"
NUM_ACTIVE_LIGHTS = 3

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
np.random.seed(42)
tf.random.set_seed(42)

In [17]:
df = pd.read_csv(CSV_PATH, low_memory=False)

filtered = df[
    df["light_folder"].isin(ALLOWED_LIGHT_FOLDERS)
    & (df["num_active_lights"].astype(int) == NUM_ACTIVE_LIGHTS)
    & (df["material_folder"].astype(str) == MATERIAL)
    & (df["batch_folder"].astype(str) == BATCH)
].copy()

cols = [
    "image_relpath",
    "shape_name",
    "material_folder",
    "light_folder",
    "batch_folder",
    "frame",
    "config_id",
    "camera_png",
    "camera_name",
    "cam_pos_x", "cam_pos_y", "cam_pos_z",
    "cam_forward_x", "cam_forward_y", "cam_forward_z",
    "cam_up_x", "cam_up_y", "cam_up_z",
    "cam_right_x", "cam_right_y", "cam_right_z",
    "focal_length_mm",
    # Light 0 (Key light)
    "light0_energy", "light0_color_r", "light0_color_g", "light0_color_b",
    "light0_pos_x", "light0_pos_y", "light0_pos_z",
    "light0_dir_x", "light0_dir_y", "light0_dir_z",
    # Light 1 (Fill light)
    "light1_energy", "light1_color_r", "light1_color_g", "light1_color_b",
    "light1_pos_x", "light1_pos_y", "light1_pos_z",
    "light1_dir_x", "light1_dir_y", "light1_dir_z",
    # Light 2 (Back light)
    "light2_energy", "light2_color_r", "light2_color_g", "light2_color_b",
    "light2_pos_x", "light2_pos_y", "light2_pos_z",
    "light2_dir_x", "light2_dir_y", "light2_dir_z",
]

filtered_cols = filtered[cols].copy()

In [18]:
# convert to camera local coordinates with blender conventions

def normalize_rows(v: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.clip(n, eps, None)

def add_lights_cam_coords_blender_local(df):
    cam_pos = df[["cam_pos_x", "cam_pos_y", "cam_pos_z"]].to_numpy(dtype=np.float32)

    cam_right = normalize_rows(df[["cam_right_x", "cam_right_y", "cam_right_z"]].to_numpy(dtype=np.float32))
    cam_up    = normalize_rows(df[["cam_up_x",    "cam_up_y",    "cam_up_z"]].to_numpy(dtype=np.float32))

    cam_forward = normalize_rows(df[["cam_forward_x", "cam_forward_y", "cam_forward_z"]].to_numpy(dtype=np.float32))
    cam_back = -cam_forward

    # Process each of 3 lights
    for light_idx in range(3):
        light_pos = df[[f"light{light_idx}_pos_x", f"light{light_idx}_pos_y", f"light{light_idx}_pos_z"]].to_numpy(dtype=np.float32)
        light_dir = df[[f"light{light_idx}_dir_x", f"light{light_idx}_dir_y", f"light{light_idx}_dir_z"]].to_numpy(dtype=np.float32)
        light_dir = normalize_rows(light_dir)

        rel = light_pos - cam_pos

        df[f"light{light_idx}_pos_cam_x"] = np.einsum("ij,ij->i", rel, cam_right)
        df[f"light{light_idx}_pos_cam_y"] = np.einsum("ij,ij->i", rel, cam_up)
        df[f"light{light_idx}_pos_cam_z"] = np.einsum("ij,ij->i", rel, cam_back)

        df[f"light{light_idx}_dir_cam_x"] = np.einsum("ij,ij->i", light_dir, cam_right)
        df[f"light{light_idx}_dir_cam_y"] = np.einsum("ij,ij->i", light_dir, cam_up)
        df[f"light{light_idx}_dir_cam_z"] = np.einsum("ij,ij->i", light_dir, cam_back)

    return df

cols_prepped = add_lights_cam_coords_blender_local(filtered_cols.copy())

In [19]:
image_df = cols_prepped.copy()
image_df["image_path"] = image_df["image_relpath"].astype(str).apply(lambda p: os.path.join(DATASET_ROOT, p))

image_df = image_df[image_df["image_path"].map(os.path.exists)].reset_index(drop=True)
if image_df.empty:
    raise ValueError("No images found for current filters. Check DATASET_ROOT and image_relpath values.")

def load_and_preprocess_image(path: str) -> np.ndarray:
    with Image.open(path) as img:
        img = img.convert("RGB").resize(IMG_SIZE, Image.BILINEAR)
        return np.asarray(img, dtype=np.float32) / 255.0

X_images = np.stack([load_and_preprocess_image(p) for p in image_df["image_path"]], axis=0)

In [20]:
# Target columns: all 3 lights, position and direction in camera space
target_cols = []
for light_idx in range(3):
    target_cols.extend([
        f"light{light_idx}_pos_cam_x", f"light{light_idx}_pos_cam_y", f"light{light_idx}_pos_cam_z",
        f"light{light_idx}_dir_cam_x", f"light{light_idx}_dir_cam_y", f"light{light_idx}_dir_cam_z",
    ])

# make sure no leakage from labeled features
exclude_cols = set(target_cols + [
    "image_relpath", "image_path", "camera_png",
])
# Also exclude world-space light coordinates
for light_idx in range(3):
    exclude_cols.update([
        f"light{light_idx}_pos_x", f"light{light_idx}_pos_y", f"light{light_idx}_pos_z",
        f"light{light_idx}_dir_x", f"light{light_idx}_dir_y", f"light{light_idx}_dir_z",
    ])

known_df = image_df[[c for c in image_df.columns if c not in exclude_cols]].copy()
cat_cols = [c for c in ["shape_name", "material_folder", "light_folder", "batch_folder", "camera_name"] if c in known_df.columns]
known_df = pd.get_dummies(known_df, columns=cat_cols, drop_first=False)

X_known = known_df.to_numpy(dtype=np.float32)
y = image_df[target_cols].to_numpy(dtype=np.float32)

idx = np.arange(len(image_df))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42)
idx_train, idx_val = train_test_split(idx_train, test_size=0.2, random_state=42)

X_img_train, X_img_val, X_img_test = X_images[idx_train], X_images[idx_val], X_images[idx_test]
X_tab_train, X_tab_val, X_tab_test = X_known[idx_train], X_known[idx_val], X_known[idx_test]
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]

tab_mean = X_tab_train.mean(axis=0, keepdims=True)
tab_std = X_tab_train.std(axis=0, keepdims=True)
tab_std[tab_std < 1e-8] = 1.0

X_tab_train = (X_tab_train - tab_mean) / tab_std
X_tab_val = (X_tab_val - tab_mean) / tab_std
X_tab_test = (X_tab_test - tab_mean) / tab_std

print("Train/Val/Test:", len(idx_train), len(idx_val), len(idx_test))
print("Image input shape:", X_img_train.shape[1:])
print("Tabular features:", X_tab_train.shape[1])
print("Target outputs (18 = 3 lights * 6 values):", y_train.shape[1])

Train/Val/Test: 848 212 266
Image input shape: (224, 224, 3)
Tabular features: 37
Target outputs (18 = 3 lights * 6 values): 18


In [21]:
# Build model with larger output layer for 3 lights (18 outputs instead of 6)
img_input = keras.Input(shape=(*IMG_SIZE, 3), name="image")
x = layers.Conv2D(32, 3, activation="relu", padding="same")(img_input)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)

tab_input = keras.Input(shape=(X_tab_train.shape[1],), name="known_params")
t = layers.Dense(128, activation="relu")(tab_input)
t = layers.Dense(64, activation="relu")(t)

h = layers.Concatenate()([x, t])
h = layers.Dense(256, activation="relu")(h)
h = layers.Dropout(0.3)(h)
h = layers.Dense(128, activation="relu")(h)
out = layers.Dense(18, name="lights_cam_pose")(h)  # 3 lights * 6 outputs

model = keras.Model(inputs=[img_input, tab_input], outputs=out)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
model.summary()

callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]

history = model.fit(
    {"image": X_img_train, "known_params": X_tab_train},
    y_train,
    validation_data=({"image": X_img_val, "known_params": X_tab_val}, y_val),
    epochs=30,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 224, 224,  │        896 │ image[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 112, 112,  │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 112, 112,  │     18,496 │ max_pooling2d_2[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 56, 56,    │          0 │ conv2d_4[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 56, 56,    │     73,856 │ max_pooling2d_3[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ conv2d_5[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ known_params        │ (None, 37)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 256)       │     33,024 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 128)       │      4,864 │ known_params[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 256)       │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 64)        │      8,256 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 320)       │          0 │ dropout_2[0][0],  │
│ (Concatenate)       │                   │            │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 256)       │     82,176 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 256)       │          0 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 128)       │     32,896 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lights_cam_pose     │ (None, 18)        │      2,322 │ dense_9[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 256,786 (1003.07 KB)

 Trainable params: 256,786 (1003.07 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 19s 658ms/step - loss: 4.0468 - mae: 1.3793 - val_loss: 1.4268 - val_mae: 0.8717
Epoch 2/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 18s 650ms/step - loss: 0.8026 - mae: 0.6553 - val_loss: 0.3503 - val_mae: 0.4163
Epoch 3/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 18s 653ms/step - loss: 0.3505 - mae: 0.4359 - val_loss: 0.0939 - val_mae: 0.2266
Epoch 4/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 18s 652ms/step - loss: 0.1816 - mae: 0.3219 - val_loss: 0.0356 - val_mae: 0.1454
Epoch 5/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 18s 660ms/step - loss: 0.1413 - mae: 0.2819 - val_loss: 0.0233 - val_mae: 0.1127
Epoch 6/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 18s 657ms/step - loss: 0.1305 - mae: 0.2661 - val_loss: 0.0221 - val_mae: 0.1108
Epoch 7/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 18s 668ms/step - loss: 0.1111 - mae: 0.2449 - val_loss: 0.0191 - val_mae: 0.1027
Epoch 8/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 18s 656ms/step - loss: 0.1050 - mae: 0.2359 - val_loss: 0.0166 - val_mae: 0.0944
Epoch 9/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 18s 660ms/

In [22]:
test_loss, test_mae = model.evaluate({"image": X_img_test, "known_params": X_tab_test}, y_test, verbose=0)
y_pred = model.predict({"image": X_img_test, "known_params": X_tab_test}, verbose=0)

# Evaluate per-light metrics
print(f"Test loss: {test_loss:.6f}")
print(f"Test MAE (all 18 outputs): {test_mae:.6f}")
print()

light_names = ["Key (Light 0)", "Fill (Light 1)", "Back (Light 2)"]
for light_idx in range(3):
    start_idx = light_idx * 6
    end_idx = start_idx + 6
    
    y_pred_light = y_pred[:, start_idx:end_idx]
    y_test_light = y_test[:, start_idx:end_idx]
    
    # Position error
    pos_mae = np.mean(np.abs(y_pred_light[:, :3] - y_test_light[:, :3]))
    
    # Direction error
    pred_dir = y_pred_light[:, 3:6]
    true_dir = y_test_light[:, 3:6]
    pred_dir = pred_dir / np.clip(np.linalg.norm(pred_dir, axis=1, keepdims=True), 1e-8, None)
    true_dir = true_dir / np.clip(np.linalg.norm(true_dir, axis=1, keepdims=True), 1e-8, None)
    cosang = np.sum(pred_dir * true_dir, axis=1)
    cosang = np.clip(cosang, -1.0, 1.0)
    angle_err_deg = np.degrees(np.arccos(cosang))
    
    print(f"{light_names[light_idx]}:")
    print(f"  Position MAE (camera xyz): {pos_mae:.6f}")
    print(f"  Direction angular error mean (deg): {angle_err_deg.mean():.3f}")
    print(f"  Direction angular error median (deg): {np.median(angle_err_deg):.3f}")
    print()

Test loss: 0.009946
Test MAE (all 18 outputs): 0.075618

Key (Light 0):
  Position MAE (camera xyz): 0.102516
  Direction angular error mean (deg): 5.000
  Direction angular error median (deg): 4.814

Fill (Light 1):
  Position MAE (camera xyz): 0.095786
  Direction angular error mean (deg): 3.237
  Direction angular error median (deg): 3.224

Back (Light 2):
  Position MAE (camera xyz): 0.106105
  Direction angular error mean (deg): 4.852
  Direction angular error median (deg): 4.505



In [23]:
# save model
model.save("tri_angular_predictor.keras")

In [ ]:
# Ttest on a single unseen image from a different batch
test_image_path = os.path.join(DATASET_ROOT, "Cylinder", "PlasticGlossy", "Tri Lighting", "Batch 2 - Cycles AGX Punchy", "1.png")
if not os.path.exists(test_image_path):
    raise FileNotFoundError(f"Image not found: {test_image_path}")

# build image input
test_image = load_and_preprocess_image(test_image_path)
X_img_single = np.expand_dims(test_image, axis=0)

# match csv row, then convert to camera coordinates with earlier function
test_relpath = os.path.relpath(test_image_path, DATASET_ROOT).replace("\\", "/")
row_match = df[df["image_relpath"].astype(str).str.replace("\\", "/", regex=False) == test_relpath].copy()
if row_match.empty:
    raise ValueError(f"No CSV row found for image_relpath={test_relpath}")
if len(row_match) > 1:
    print(f"Found {len(row_match)} rows for this image_relpath; using the first match.")
row_match = row_match.iloc[[0]].copy()

row_cam = add_lights_cam_coords_blender_local(row_match[cols].copy())
y_true_single = row_cam[target_cols].iloc[0].to_numpy(dtype=np.float32)

# build known-parameter input exactly like training
known_single = row_cam[[c for c in row_cam.columns if c not in exclude_cols]].copy()
single_cat_cols = [c for c in cat_cols if c in known_single.columns]
known_single = pd.get_dummies(known_single, columns=single_cat_cols, drop_first=False)
known_single = known_single.reindex(columns=known_df.columns, fill_value=0.0)

X_tab_single = known_single.to_numpy(dtype=np.float32)
X_tab_single = (X_tab_single - tab_mean) / tab_std
if X_tab_single.shape[0] != X_img_single.shape[0]:
    raise ValueError(f"Batch mismatch: image batch={X_img_single.shape[0]}, tabular batch={X_tab_single.shape[0]}")

# predict
y_pred_single = model.predict([X_img_single, X_tab_single], verbose=0)[0]

# compare prediction vs converted camera-space truth for each light
print("Image:", test_relpath)
print()

for light_idx in range(3):
    start_idx = light_idx * 6
    end_idx = start_idx + 6
    
    pred_vals = y_pred_single[start_idx:end_idx]
    true_vals = y_true_single[start_idx:end_idx]
    
    pred_pos = pred_vals[:3]
    true_pos = true_vals[:3]
    pred_dir = pred_vals[3:6]
    true_dir = true_vals[3:6]
    
    pred_dir = pred_dir / np.clip(np.linalg.norm(pred_dir), 1e-8, None)
    true_dir = true_dir / np.clip(np.linalg.norm(true_dir), 1e-8, None)
    cosang = np.clip(np.dot(pred_dir, true_dir), -1.0, 1.0)
    angle_err_deg = float(np.degrees(np.arccos(cosang)))
    pos_abs_err = np.abs(pred_pos - true_pos)
    
    light_names_test = ["Key", "Fill", "Back"]
    print(f"{light_names_test[light_idx]} Light:")
    print(f"  Predicted pos (cam xyz): {pred_pos}")
    print(f"  True pos (cam xyz):      {true_pos}")
    print(f"  Abs pos error (xyz):     {pos_abs_err}")
    print(f"  Predicted dir (cam xyz): {pred_dir}")
    print(f"  True dir (cam xyz):      {true_dir}")
    print(f"  Direction angle error (deg): {angle_err_deg:.3f}")
    print()

Found 13 rows for this image_relpath; using the first match.
Image: Cylinder/PlasticGlossy/Tri Lighting/Batch 2 - Cycles AGX Punchy/1.png

Key Light:
  Predicted pos (cam xyz): [-2.2893183  4.0468545  1.0066313]
  True pos (cam xyz):      [-2.1212986  3.962853   0.8795951]
  Abs pos error (xyz):     [0.16801977 0.08400154 0.12703615]
  Predicted dir (cam xyz): [ 0.21136874 -0.752921   -0.62324405]
  True dir (cam xyz):      [ 0.3213995  -0.723098   -0.61141783]
  Direction angle error (deg): 6.570

Fill Light:
  Predicted pos (cam xyz): [-3.8830674  4.0477467 -6.487777 ]
  True pos (cam xyz):      [-3.726979  3.96285  -6.204559]
  Abs pos error (xyz):     [0.15608835 0.08489656 0.28321838]
  Predicted dir (cam xyz): [ 0.61539525 -0.69878334  0.36467883]
  True dir (cam xyz):      [ 0.5536124  -0.72309774  0.41308966]
  Direction angle error (deg): 4.709

Back Light:
  Predicted pos (cam xyz): [3.5959265  4.031211   0.21945328]
  True pos (cam xyz):      [3.3749774  3.9628527  0.1295545